# 01 — Vectors, matrices, the dot product

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/001-vectors-matrices-dot-product.ipynb)

Every score in this course is a dot product: multiply matching positions, add the results. In this notebook you compute one by hand, plot how it changes as the weight turns, and drag a slider to feel what the numbers mean. Everything runs on a fresh Colab runtime.

## 1. Two lists of numbers

A vector is one data point written as a list. Here the data has two features, and the model has two weights — one per feature. The dot product multiplies each feature by its weight and adds the two products.

In [ ]:
import numpy as np                          # the array library, preinstalled on Colab

x = np.array([2., 1.])                     # one data point: feature 1 = 2, feature 2 = 1
w = np.array([3., -1.])                    # the weights the model learned
b = -1.                                    # the bias: a constant added after the dot product

score = x @ w + b                          # @ is the dot product; then add the bias
print("x . w =", x @ w)                     # 5.0
print("score =", score)                     # 4.0

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `x = ...` — the input: two numbers, one per feature.
- `w = ...` — the learned weights, same length as `x`.
- `b = -1.` — a constant that shifts the result after the dot product.
- `score = x @ w + b` — `@` means dot product; the bias is added afterwards.

## 2. The same thing, by hand

The `@` symbol hides the arithmetic. Do it one term at a time: each term is one feature's contribution, and the sum is the score.

In [ ]:
total = 0.                                 # start the running total at zero

for i in range(len(x)):                    # visit feature 0, then feature 1
    term = x[i] * w[i]                     # multiply matching positions
    total = total + term                   # add this feature's contribution
    print(f"feature {i}: {x[i]} x {w[i]} = {term}")   # show each term

print("dot product =", total)              # 5.0, same answer as section 1

**Line by line**

- `total = 0.` — the accumulator for the sum.
- `for i in range(len(x))` — one iteration per feature.
- `term = x[i] * w[i]` — the contribution of feature `i`.
- `total = total + term` — add it to the running total.
- `print(f"...")` — an f-string; the braces are replaced by values.

## 3. Plot: the score as the weight turns

Keep the data fixed and rotate the weight. The score is the data's shadow on the weight, so it peaks when they point the same way and hits zero at 90°.

In [ ]:
import matplotlib.pyplot as plt            # plotting library, preinstalled on Colab

angles = np.arange(0, 181, 5)              # 0 to 180 degrees, step 5
x_fixed = np.array([2., 0.])               # the data stays fixed on the x-axis
scores = []                                # collect one score per angle

for a in angles:                           # walk through the angles
    th = np.deg2rad(a)                     # convert degrees to radians
    w_a = np.array([np.cos(th), np.sin(th)])   # a unit weight at this angle
    scores.append(x_fixed @ w_a)           # dot product with the fixed data

plt.figure(figsize=(6, 3))                 # a wide, short figure
plt.plot(angles, scores, color="tab:red")  # score against angle
plt.axhline(0, color="grey", linewidth=0.8)   # the zero line
plt.axvline(90, color="grey", linestyle="--")  # the perpendicular angle
plt.xlabel("angle of w (degrees)")         # x-axis label
plt.ylabel("dot product")                  # y-axis label
plt.title("90 degrees -> score is zero")   # the takeaway
plt.show()                                 # render the figure

**Line by line**

- `np.arange(0, 181, 5)` — a list of angles from 0 to 180.
- `x_fixed` — the data vector, held constant across the loop.
- `np.deg2rad(a)` — NumPy's cosine expects radians, not degrees.
- `w_a = ...` — a unit-length weight pointing at angle `a`.
- `scores.append(...)` — store one number per angle.
- `plt.plot(...)` — draw the curve; the rest only labels it.

## 4. Interactive: turn the weight yourself

Drag the slider. The grey arrow is the data, the red arrow is the weight, and the title shows the score. Watch it fall to zero at 90° and turn negative after that.

In [ ]:
from ipywidgets import interact           # sliders that rerun a function

def show(angle=40):                        # angle comes from the slider
    th = np.deg2rad(angle)                 # degrees -> radians
    w_a = 2 * np.array([np.cos(th), np.sin(th)])   # weight, length 2
    x_fixed = np.array([2., 0.])           # the same fixed data
    score = x_fixed @ w_a                  # the dot product

    plt.figure(figsize=(5, 3))             # fresh figure for each drag
    plt.quiver(0, 0, *x_fixed, angles="xy", scale_units="xy", scale=1,
               color="tab:grey", label="x (data)")        # data arrow
    plt.quiver(0, 0, *w_a, angles="xy", scale_units="xy", scale=1,
               color="tab:red", label="w (weight)")       # weight arrow
    plt.xlim(-1, 3); plt.ylim(-3, 3)       # fixed axes, so nothing jumps
    plt.legend(loc="upper left")           # name both arrows
    plt.title(f"score = {score:.2f}")      # live readout
    plt.show()                             # draw it

interact(show, angle=(0, 180, 5))          # attach the slider

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(angle=40)` — the function that redraws one frame.
- `w_a = 2 * ...` — a weight vector of length 2 at the slider's angle.
- `score = x_fixed @ w_a` — recompute the dot product.
- `plt.quiver(...)` — an arrow from the origin; `scale=1` keeps units honest.
- `interact(show, angle=(0,180,5))` — the slider, its range and step.

## 5. Dot product vs cosine

Two documents can point the same way but have different lengths. The dot product mixes length with direction; cosine divides both lengths out. Compare them on one query.

In [ ]:
q = np.array([1., 0.])                      # the query
docs = {"A": np.array([50., 0.]),           # a long document, same direction
        "B": np.array([1., .1])}            # a short document, almost the same direction

names, dots, cosines = [], [], []           # three parallel lists
for name, v in docs.items():               # one iteration per document
    d = q @ v                               # raw dot product
    c = d / (np.linalg.norm(q) * np.linalg.norm(v))   # cosine: lengths removed
    names.append(name); dots.append(d); cosines.append(c)   # store results
    print(f"{name}: dot={d:.2f}  cosine={c:.3f}")        # compare

plt.figure(figsize=(5, 3))                 # bar chart of both metrics
plt.bar([n + " dot" for n in names], dots, color="tab:orange")       # raw scores
plt.bar([n + " cos" for n in names], cosines, color="tab:green")     # cosine scores
plt.axhline(1, color="grey", linestyle="--")   # cosine ceiling
plt.title("dot prefers length; cosine ignores it")   # takeaway
plt.show()                                 # render

**Line by line**

- `q = ...` — the query vector.
- `docs = {...}` — a dictionary: name → vector.
- `np.linalg.norm(v)` — the length of a vector.
- `c = d / (...)` — cosine = dot ÷ (length × length).
- `plt.bar(...)` — two bars per document, raw and normalised.

## Your turn

Change `x_fixed` in section 4 to `[0., 2.]` and predict the score at 0° before you rerun. Then make the cosine of document B larger than A by shortening A — what has to change?